In [1]:
import numpy as np

# =====================================================================
# 1. PARAMETERS
# =====================================================================
# Note: These are placeholder values. For gold/silver clusters, L0 will 
# be significantly larger compared to organic molecules.
N = 4 # Number of sites

# Geometric Parameters (in nanometers)
a = 0.4 # Helical radius
c_pitch = 1.2 # Total length of the helix
turns = 1.0 # Number of full helical turns

# Static Energy Parameters (in eV)
e0 = 0.0 # On-site energy
t0 = 0.05 # Nearest-neighbor hopping
L0 = 0.01 # Next-nearest-neighbor Spin-Orbit Coupling (SOC)

# Dynamic Coupling Strengths (in eV)
e_nu = 0.0 # Holstein on-site modulation 
t_nu = 0.005 # Hopping modulation
L_nu = 0.001 # SOC modulation

# Pauli Matrices
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I_2 = np.eye(2, dtype=complex)

# =====================================================================
# 2. GENERATE 3D GEOMETRY & CHIRALITY VECTORS
# =====================================================================
r = np.zeros((N, 3))
for m in range(N):
    phi = 2 * np.pi * turns * (m / (N - 1))
    z = c_pitch * (m / (N - 1))
    r[m] = [a * np.cos(phi), a * np.sin(phi), z]

# Calculate normalized bond vectors between adjacent sites
d = np.zeros((N-1, 3))
for m in range(N-1):
    bond = r[m+1] - r[m]
    d[m] = bond / np.linalg.norm(bond)

# Calculate geometric chirality vectors (cross product of adjacent bonds)
# v[m] is defined for m = 0 to N-3 (next-nearest neighbors)
v = np.zeros((N-2, 3))
for m in range(N-2):
    v[m] = np.cross(d[m], d[m+1])

# =====================================================================
# 3. CONSTRUCT 8x8 SYSTEM & COUPLING MATRICES (Fock State Basis)
# =====================================================================
# State ordering: |1,up>, |1,down>, |2,up>, |2,down>, ...
H_S = np.zeros((2*N, 2*N), dtype=complex)
S_nu = np.zeros((2*N, 2*N), dtype=complex)

# A. Populate On-Site Energies (Diagonal blocks)
for m in range(N):
    idx = 2 * m
    H_S[idx:idx+2, idx:idx+2] = e0 * I_2
    S_nu[idx:idx+2, idx:idx+2] = e_nu * I_2

# B. Populate Nearest-Neighbor Hopping (1st off-diagonal blocks)
for m in range(N-1):
    idx1 = 2 * m # Current site
    idx2 = 2 * (m + 1) # Next site
    
    # Hop forward (m to m+1)
    H_S[idx1:idx1+2, idx2:idx2+2] = -t0 * I_2
    S_nu[idx1:idx1+2, idx2:idx2+2] = -t_nu * I_2
    
    # Hop backward (m+1 to m) - Hermitian conjugate
    H_S[idx2:idx2+2, idx1:idx1+2] = -t0 * I_2
    S_nu[idx2:idx2+2, idx1:idx1+2] = -t_nu * I_2

# C. Populate Next-Nearest-Neighbor SOC (2nd off-diagonal blocks)
for m in range(N-2):
    idx1 = 2 * m # Current site
    idx2 = 2 * (m + 2) # Next-nearest site
    
    # Construct the v dot sigma term for this specific spatial hop
    v_dot_sigma = v[m, 0]*sx + v[m, 1]*sy + v[m, 2]*sz
    
    # Static SOC block for System Hamiltonian
    V_static = 1j * L0 * v_dot_sigma
    H_S[idx1:idx1+2, idx2:idx2+2] = V_static
    H_S[idx2:idx2+2, idx1:idx1+2] = V_static.conj().T
    
    # Dynamic SOC block for System-Bath Coupling
    V_dynamic = 1j * L_nu * v_dot_sigma
    S_nu[idx1:idx1+2, idx2:idx2+2] = V_dynamic
    S_nu[idx2:idx2+2, idx1:idx1+2] = V_dynamic.conj().T

# =====================================================================
# 4. DIAGONALIZE & TRANSFORM INTO ENERGY EIGENBASIS
# =====================================================================
# Find the energy eigenbasis of the system Hamiltonian
# evals = energy levels, evecs = unitary transformation matrix (U)
evals, evecs = np.linalg.eigh(H_S)

# Project the coupling operator into the energy eigenbasis
# S_tilde_jk = <j| S_nu |k>
S_tilde = evecs.conj().T @ S_nu @ evecs

# =====================================================================
# 5. VERIFICATION OUTPUT
# =====================================================================
print("--- Geometric Chirality Vectors ---")
for m in range(N-2):
    print(f"Site {m+1} to {m+3}: {np.round(v[m], 3)}")

print("\n--- Energy Eigenvalues (eV) ---")
print(np.round(evals, 5))

print("\n--- Diagonal Elements of S_tilde (Drives Pure Dephasing, T2*) ---")
# The diagonal elements dictate how the bath shifts each energy level
print(np.round(np.diag(S_tilde).real, 5))


--- Geometric Chirality Vectors ---
Site 1 to 3: [0.65  0.375 0.65 ]
Site 2 to 4: [-0.65   0.375  0.65 ]

--- Energy Eigenvalues (eV) ---
[-0.08237 -0.08237 -0.0302  -0.0302   0.0302   0.0302   0.08237  0.08237]

--- Diagonal Elements of S_tilde (Drives Pure Dephasing, T2*) ---
[-0.00824 -0.00824 -0.00302 -0.00302  0.00302  0.00302  0.00824  0.00824]
